# Week 2
### Latent variable inference

In Week 2, your task is to formulate the two models as (time-homogeneous) **Hidden Markov Models (HMM)** with *discrete* states. To recap, <br>
a HMM is described by latent variables that are (1) temporal/sequential and (2) form a **Markov chain (MC)**.<br> 
If we denote these variable at time step $t$ by $x_t$, the Markovian property means that conditioned on $x_t$, all future states<br>
$(x_{t+1}, x_{t+2}, \ldots)$ are independent of states at times before $t$.<br>
Colloquially, conditioned on the present, the future is independent of the past.<br> 

In an HMM, the Markovian state variables are not directly observerd. Instead at each time, we observe an observed variable, $n_t$, which<br>
only depends of the Markov state, $x_t$, at the same time step. For us this dependence is given by the Poisson distribution describing the <br>
spike emissions, conditioned on the rates, with the latter being determined by $x_t$ (in other words, the rate at time $t$ is a<br>
deterministic instantaneous functions of $x_t$).<br> Figure 2 shows the graphical model for the HMM, with such Poisson observation (aka emission) distributions.  

Here, we aim to design Markov chains with discrete states to approximate the behavior of the latent variables of the step and ramp models.<br>
In the case of the step model, the discretization is actually exact, as the model really just has two levels of rates (although we will see that the correspondence between Markov states and possible firing rate levels is rather complicated). <br>
It is for the ramp model  that a discretization will be an approximation (since in the original formulation of this model, the state variables, $x_t$, are continuous variables). 

The reason for formulating the models as discrete state HMMs is that in this case we can use the powerful and efficient 
[forward-backward algorithm](https://en.wikipedia.org/wiki/Forward%E2%80%93backward_algorithm) to calculate<br>
(1) the Bayesian *posterior* estimate (the posterior mean) of the state variables, $x_t$, given the observations, $n_t$, and<br>
(2) calculate the model likelihood function,  $P(n_{1:t}|\Theta, \mathcal{M})$, i.e. the probability of the observed spike train conditioned on the model parameters $\Theta$ (for each model $\mathcal{M} = $ ramp or step).


<img src="figs/HMM_graph.png" width=600/>





### Task 2.1: 
**Forming an HMM approximation to the ramp model**

$\newcommand{\T}{\mathcal{T}}$

The state variable of the (original) ramp model is continuous and, because the update rule $x_{t+1} = x_t + \beta dt + \sigma\sqrt{dt}\epsilon_t$<br>
involves the Gaussian variables $\epsilon_t$, their transition probabilities $P(x_{t+1}| x_{t})$ are Gaussian (with mean and variance possibly depending on $x_t$, as well as the model parameters). Your first sub-task is to work out this distribution.

(Done by parker in task 2 notebook)

First, we will now assume $x_t$ does not go below 0. Thus if $x_t$ is currently zero, and the proposed change according to normal update rule of $x_t$ is negative, $x_t$ will remain at 0; only if the proposed change is positive it is actually implemented. (Note that this is a redefinition of the model's latent variables, it does not affect the behaviour of rates and thus spikes; and thus it does really change the model.)

(Boundary conditions on change in $x$)

Next, we will approximate $x_t$ by assuming it takes values on a regular grid of $K$ points going from 0 to 1, inclusive of both ends. Let's use $K=50$ or $K=100$.<br>
These $K$ points form the discrete states of the discrete-state HMM that approximates the original ramp model. We will denote these states by their index $s$ going from 0 to $K-1$.
The corresponding value of $x_t$ is then given by 

$x_t = \frac{s_t}{K-1} \qquad \text{where} \quad s_t \in \{0, \ldots, K - 1\}$.

By evaluating the relevant probabilities, based on the Gaussian distribution you derived above, form the transition matrix for the Markov chain, defined as 

$
\T_{s,s'} = P(s_{t+1}= s'| s_{t}= s)
$

Note that the state *transitioned to* (i.e. the one at $t+1$) is the column index of the matrix. Thus the rows of $\T$ have to sum up to 1.<br> 
You may need to (and it certainly would not hurt to) **enforce this constraint by hand** after constructing the (intial) matrix using the evaluated Gaussian probabilities. <br>

Also note transitions out of the last state $s = K-1$ need special consideration, as according to the original model, once the variable $x$ reaches 1, it stays there. 

Next, you will need to form the initial state distribution, $\pi$, as an array of $K$ values (summing to 1!) giving the probabilities of different possibilities of $s_0$. This should approximate the equation $x[0] = x_0 + \sigma\sqrt{dt} \epsilon_0$.

Once the transition matrix $\T$ and initial state distribution $\pi$ are formed, we can simulate the chain. In order to <br> do this, you will first draw $s_0$ from the initial state distribution, then successively sample from the <br>
appropriate distribution according to the transition matrix $\T$ (and depending on the current state $s_t$). To sample <br>
the discrete (integer) $s$ from a distribution over its $K$ possibilities, you can use `np.random.choice`.

For different choices of the model parameters, $\beta$, $\sigma$ and $x_0$, simulate several trials of this chain and (after appropriate rescaling) plot the trajectories $x_t$. Based on the trajectory $x_t$, calculate the firing rate trajectory $r_t$. Compare these rate trajectories with corresponding simulated rate trajectories of the original (continuous state) ramp model, to make sure your implementation is accurate *enough*. 

Note that for small enough values of $\sigma$ the Markov chain approximation will produce trajectories that get stuck at the inital state. Why is this? For the case $\beta=0$, estimate, solely in terms of $K$ and $T$ (or $dt$), the *order of magnitude* (or "scaling") of the value of $\sigma$  (up to a constant of proportionality) below which trajectories tend to get stuck. In the rest of the project use values of $\sigma$ above this value, unless indicated otherwise (e.g. when use of specific ranges for parameters are instructed). 

**Note:** Note also that if you use values of $\sigma$ that are too small, and depending on your code for constructing the transition matrix, your code for generating $\T$ or $\pi$ may run into numerical truncation issues resulting in `NaN` values. Something that could help is implementing things first in terms of log-probabilities, using the numerically stable function `scipy.special.logsumexp` in the normalization step (when you normalize rows of $\T$ or the vector $\pi$), and only in the end exponentiating to obtain the actual $\T$ or $\pi$. 

### Task 2.2: 
**Forming an HMM approximation to the step model**

Intuitively, given its discrete (binary) rate levels, the step model can actually be exactly formulated as a discrete state HMM.<br>

First, implement a *time-homogeneous* Markov chain representing the step state with two states. How would you choose the transition probabilities?<br> Hint: think of the 
parameter $p$ of the Negative Binomial distribution (see the Wikipedia article), which in terms of $m$ and $r$ is given by $r/(m+r)$.

Simulate the Markov chain for several trials and plot the corresponding $x_t$ trajectories. 

Also evaluate the jump times (time-steps) and make histograms of these jump times. How do the histograms appear? Do they resemble any of the histograms of jump times<br> (i.e. histograms 
for different values of $m$ and $r$) that  you made in Week 1? 

What do you think is wrong with the 2-state Markov chain approximation to the step model? **§**<br>
Read about the [Negative Binomial distribution](https://en.wikipedia.org/wiki/Negative_binomial_distribution) and its "meaning" (*for the case of positive integer $r$*), to get clues for constucting an exact Markov Chain formulation of the step model (Hint: use $r + 1$ states!).**§§**

Again, simulate several trials of this new chain, plot state trajectories, and form histograms of jump times for different values of $r$. How do these compare with the histograms you obtained in Week 1 for jump times of the original step model?


**§:** An exact fomrulation of the step model is possible as a *time-inhomogenous* 2-state Markov chain (MC). You can experiment with that of course (this is optional). However, we will be using a time-homogenous HMM, and so in the above sub-task I am asking you to construct a time-homogenous Markov chain. If you did implement a time-inhomogeneous MC formulation, feel free to write about it in your final report. But make sure you do investigate the time-homogeneous version (which is in general not a correct formulation of the original model) as well, and answer the questions for that (too).

**§§:** The suggested MC implementation of the step model will differ from the Week 1 step model, in which step time had the distribution $\mathrm{NB}(\tau | m, r)$ by a time shift by $r$ time-steps.

### Task 2.3: 
**Inference of hidden states**

Henceforth, for the rest of the activities this week (as well as most of the activities of the next two weeks), we will be solely working with the discrete-state HMM versions of the two models that you have implemented (instead of using the `models.py` simulators). We will also limit the values of the parameter $r$ to positive integers.

The `hmm_expected_states` function (see its doc/help) in the `inference.py` (run the next code cell to import this on Colab **§**) module implements the forward-backward algorithm (FBA) to calculate the posterior probabilities $P(s_t | n_{1:T})$, as well as the log-likelihood $\ln P(n_{1:T})$.**§§**<br> 
One of the inputs to `hmm_expected_states` is the array, `ll`, of the logs of the conditional observation probabilities $ll[t, s] = \log P(n_t|s_t = s)$.**§§§** Use the function `inference.poisson_logpdf` from the provided new module `inference.py`, to construct this array based on the observed spike counts of one or several trials (see the help of this function).



Use this function to obtain the posterior probabilities, $P(s_t | n_{1:T})$, for your finite-state HMM implementations of both models.**§§§§**

- Write code to calcualte the posterior expectation of $x_t$ (i.e. $\mathbb{E}[x_t | n_{1:T}]$), for the ramp model, based on the posterior probabilities $P(s_t | n_{1:T})$. Generate several trial spike trains using the discrete-state HMM ramp model (with the corresponding $x_t$ trajectories retained), and for each trial infer and plot $\mathbb{E}[x_t | n_{1:T}]$, together with the ground-truth simulated $x_t$. Repeat this in different regions of model parameter space (including low and high values of $x_0$ and $R_h$). In what parameter regimes is the inference more accurate, and vice versa? Provide intuitive/qualitative explanations for your observations. 

- Repeat the above for the step model, with the following modifications. The aspect of the step model's hidden states that we really care about is whether or not the ''neuron" has jumped to the upper rate level. Calculate the probability of being in the upper rate level based on the posterior state probabilities $P(s_t | n_{1:T})$, and again make plots of it for various simulated spike-train trials. Visualise the true jump time on these plots. You can take the time point when the posterior probability of being in the upper rate level exceeds 0.5 as the estimated/inferred jump time, and mark that on the plots as well.<br>
For your report, try to combine different trials in one plot or figure, in a compact but nice way.<br> Based on these plots, comment (include both accounts of your observations and your qualitative explanations for them) on the accuracy  of the inference in different regions of the model parameter space. 

- The `hmm_expected_states` function has an optional boolean input `filter` which is false by default. When true, the function calculates $P(s_t | n_{1:t})$ instead of $P(s_t | n_{1:T})$: i.e. the posterior conditioned only on observations up to and including the "current" time-step $t$. This corresponds to the so-called filtering problem (as in the Kalman filter; the default case, `filter = False`, corresponds to "smoothing"). Filtering is appropriate for applications where inference has to be performed online, in which case, to infer $x_t$ we do not have the luxury of having access to future observations -- without a time-machine, that is! In our case (as engineers studying the computational mechanisms in area LIP), we do of course have access to the entire spike-train; hence the default option. Nevertheless, carry out a theoretical study of the differences in inference accuracy, for both models, using smoothing vs. filtering. What qualitative differences do you observe between the inference accuracy using filtering vs. smoothing? 

In all of the above sub-tasks,  quantiatify the latent-state inference accuracy by evaluating the average error (both over trials and over time, if the latter makes sense) of the posterior estimates of the $x_t$ trajectories (for the ramp model) or the jump times (for the step model). You can then make heatmap or contour plots of these errors as a function of two parameters -- and different plots for different choices of parameter pairs. (The `tricontourf` function of `matplotlib` is very useful for this purpose.)

______________________________________________________________________________________________________________________________________
**§:** The module `inference.py` makes use of the [Numba package](http://numba.pydata.org/) to speed up computations by so-called just-in-time (JIT) compilation.  Numba can be imported in Colab, but to use it locally, you will need to install it, following these [instructions](https://numba.readthedocs.io/en/stable/user/installing.html).
Also note that JIT makes a function run slowly the first time you use it. So the first time you run a JITed function (e.g. `hmm_expected_states`) it's better to run it on a "light" test case (e.g. a single spike train, rather than several spike trains, or on a short one), and only then run it for the true use case.  


**§§:** This (log) probability depends implicitly on model parameters, hence the name (log) likelihood; it is called `normalizer` in the code for reasons having to do with the fact that it normalizes the message products involved in the FBA.

**§§§:** Note that at inference time, the $n_t$ are known and fixed, but the MC states are unknown; thus, as part of the inference procedure, we need to evaluate the observation (log-)probabilities for *all* possible states, $s$, at *every* times $t$. For more details see my [notes on the FBA](https://github.com/ahmadianlab/gg3_nda/blob/main/fwdbwd.ipynb).


**§§§§:** Note that the observed spike trains used here (and later in Weeks 3 & 4) should be generated using the discrete-state HMM version of the two models. For this, first generate the latent variables as in tasks 2.1 and 2.2, and calculate the firing rates, $r_t$, based on them. Then, given the rates, sample the spike counts, $n_t$, from the corresponding Poisson distribution (you can look up how this was done in `models.py`).
